# A stellarator run through `functional_process`, on the current API

`functional_process` ports PROCESS's stellarator models into cottax (`~/jaxgraph`): a
**declared graph** of nodes with typed ports, decomposed into blocks, each block driven
by an explicit, autodiff-visible algorithm — instead of PROCESS's own architecture, which
treats the whole pipeline as one opaque function and differentiates it by finite
differences.

This is a short walkthrough, not the full tour (see git history for a longer earlier
version). It assembles the graph for one machine — the Helias stellarator of
`tests/regression/input_files/stellarator_helias.IN.DAT` — solves it, and then spends
most of its time on one fact: **the graph is now a value**. A freshly re-assembled graph
is a new object, but it compares *equal* to the first one, and JAX's own cache is keyed
on equality, not identity. Re-assembling from scratch and solving again should therefore
compile nothing.

Everything below actually executes; runs top to bottom in a fresh kernel.

In [1]:
import jax

jax.config.update("jax_enable_x64", True)

import os
import time
from pathlib import Path

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "functional_process").is_dir())
os.chdir(REPO)

import inspect
import subprocess
import sys
from collections import Counter

import jax.numpy as jnp
import numpy as np
from cottax.blocking import Blocking

from functional_process import boundary, mda, session
from functional_process.indat import REFERENCE_INPUT_FILE, graph_for, machine_from_indat

SCRATCH = Path("/tmp/claude-1000/-home-tbogaarts-PROCESS/031c3ec8-ef1e-47a6-841c-41a7b5e49c67/scratchpad/nbwork")
SCRATCH.mkdir(parents=True, exist_ok=True)
PY = sys.executable

print(REPO, "|", jnp.zeros(1).dtype, "|", REFERENCE_INPUT_FILE, "|", PY)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


/home/tbogaarts/PROCESS/.claude/worktrees/agent-ae8399e0bd9e69d82 | float64 | tests/regression/input_files/stellarator_helias.IN.DAT | /home/tbogaarts/miniconda/envs/process_port/bin/python


## 1. Assemble the graph, and look at it

`machine_from_indat` reads only the input file's integer switches and builds a tree of
model instances; `graph_for` walks that tree into a cottax `Graph` -- a binding of node
*places* to *definitions*. Nothing runs yet.

In [2]:
machine = machine_from_indat(REFERENCE_INPUT_FILE)
graph = graph_for(machine)
print(type(machine).__name__, "->", len(graph.nodes), "nodes")
print(Counter(type(d).__name__ for d in graph.definitions.values()))

StellaratorProcess -> 154 nodes
Counter({'ImplementedFunction': 150, 'RootFind': 2, 'FixedPoint': 2})


`ImplementedFunction` is the ordinary node -- ports plus a body (`CallableNode` in the old
API). `RootFind`/`FixedPoint` are the self-loops PROCESS solves by re-running a model
until it stops moving, declared as problems rather than bodies.

One node, opened up: `In`/`Out` ports are `VarPath`s into PROCESS's own namespace, and the
function that computes them.

In [3]:
place = next(n for n in graph.nodes if n.path_str() == ".stellarator.sudo_density_limit")
node = graph.definitions[place]
print(type(node).__name__, "at", place.path_str())
print("  reads:", [v.path_str() for v in node.reads])
print("  owns :", [v.path_str() for v in node.owns])

ImplementedFunction at .stellarator.sudo_density_limit
  reads: ['.physics.b_plasma_toroidal_on_axis', '.physics.p_plasma_loss_mw', '.physics.rmajor', '.physics.rminor', '.physics.nd_plasma_electrons_vol_avg', '.physics.nd_plasma_electron_line']
  owns : ['.physics.nd_plasma_electrons_max']


### Decomposed into blocks, each driven by an algorithm chosen for it

`mda.driven_graph` cuts the raw cross-node cycles into declared problems and attaches a
driver to each; `Blocking.scc` condenses the result into strongly connected components.
Most blocks are a single node, run once, in a derived order -- only the genuinely coupled
blocks are driven by an iterative algorithm.

In [4]:
driven = mda.driven_graph(graph)
blocking = Blocking.scc(driven)
n_driven = sum(1 for t in blocking.problem_types if t is not None)
print(f"{len(blocking.blocks)} blocks, {n_driven} of them driven")
print("block sizes:", sorted(Counter(len(b) for b in blocking.blocks).items()))

144 blocks, 6 of them driven
block sizes: [(1, 138), (2, 4), (3, 1), (7, 1)]


And the boundary -- every read that is not produced by another node in the graph.
`arrays are refused in the graph` (`CLAUDE.md`): what used to be `carried` fields is now
a `stated` port, an output of a source node living in the env rather than baked into a
declaration -- a third boundary category alongside `input` (a genuine external read) and
`guess` (a `Start` port minted for a driven unknown).

In [5]:
rows = boundary.boundary(driven)
print(Counter(kind for kind, _ in rows))

Counter({'input': 289, 'stated': 16, 'guess': 6})


## 2. Solve it, against PROCESS

`session.open_session(..., mode="provider")` runs PROCESS's own `SingleRun` once (cached
to disk after the first time) to get both a cold starting `DataStructure` and PROCESS's
own converged answer to compare against. `Session.mdf` assembles the MDF arm on first
call and solves it -- PROCESS's own architecture (optimise the design, converge the
coupled models inside every evaluation), same SQP (`pyvmcon`), same convergence test,
gradients from one `jax.jacfwd` instead of a finite-difference pipeline sweep per
iteration variable.

In [6]:
import contextlib
import io

from process.core.solver.objectives import objective_function

with contextlib.redirect_stdout(io.StringIO()):   # PROCESS's own console noise
    live = session.open_session(REFERENCE_INPUT_FILE, mode="provider")
reference = live.reference
print(f"PROCESS: {reference.solver_iterations} VMCON iterations in "
      f"{reference.solve_seconds:.1f} s")

began = time.perf_counter()
answer = live.mdf()
first_solve_seconds = time.perf_counter() - began
print(f"port MDF: {answer['iterations']} SQP iterations in {first_solve_seconds:.1f} s")

process_objf = objective_function(reference.i_figure_merit, reference.data)
port_objf = answer["objf"]
print(f"\nobjf   port {port_objf:.9f}   PROCESS {process_objf:.9f}   "
      f"rel {abs(port_objf - process_objf) / abs(process_objf):.2e}")

worst = max(
    abs(x - reference.converged[i]) / abs(reference.converged[i])
    for i, x in zip(reference.ixc, answer["_x"])
)
print(f"worst relative deviation on any of the 8 design variables: {worst:.2e}")

PROCESS: 46 VMCON iterations in 98.4 s
port MDF: 41 SQP iterations in 19.3 s

objf   port 1.218482841   PROCESS 1.214916785   rel 2.94e-03
worst relative deviation on any of the 8 design variables: 1.09e-01


The objective agrees closely; the design vector mostly does too. (One variable sits on a
kink in the model -- a clamped square root where PROCESS's finite difference sees a wide
chord and autodiff sees the exact one-sided slope -- already diagnosed, not a port
defect. Not chased further here.)

## 3. The centrepiece: re-assemble from scratch, and watch it compile nothing

`graph_for(machine_from_indat(...))` run twice gives two different Python objects. Are
they the same graph? Under the old API this question needed an explicit `==`; under the
new one it also decides whether JAX's own executable cache -- keyed by equality, not
identity -- treats the second assembly as new work at all.

In [7]:
machine_a = machine_from_indat(REFERENCE_INPUT_FILE)
machine_b = machine_from_indat(REFERENCE_INPUT_FILE)
graph_a, graph_b = graph_for(machine_a), graph_for(machine_b)
print("equal:", graph_a == graph_b, "  same object:", graph_a is graph_b)

equal: True   same object: False


Measured in a **fresh subprocess** (so "first assembly" is genuinely cold, not warmed by
section 2's own solve above): assemble MDF, seed, prime and solve; then do the whole
thing again **from scratch** -- a second `graph_for(machine_from_indat(...))`, a second
`build_mdf`, on the same graph *content* but not the same objects.

In [8]:
gate_script = SCRATCH / "reassembly_gate.py"
gate_script.write_text(r'''
import time
import jax
jax.config.update("jax_enable_x64", True)
from functional_process import session
from functional_process.run_cold_matrix import build_mdf, solve_mdf
from functional_process.session import _Compiles
from functional_process.indat import REFERENCE_INPUT_FILE, graph_for, machine_from_indat

live = session.open_session(REFERENCE_INPUT_FILE)
reference, cold = live.reference, live.cold

def assemble_and_solve():
    machine_graph = graph_for(machine_from_indat(REFERENCE_INPUT_FILE))
    build = build_mdf(reference, machine_graph, None, root_find=False)
    return solve_mdf(build, reference, cold)

xs = []
for label in ("first assembly", "re-assembled from scratch"):
    compiles = _Compiles()
    began = time.perf_counter()
    result = assemble_and_solve()
    n_compiles = compiles.stop()
    seconds = time.perf_counter() - began
    xs.append(result["_x"])
    print("ROW", label, seconds, n_compiles)
print("SAME", xs[0] == xs[1])
''')

out = subprocess.run([PY, str(gate_script)], cwd=REPO, capture_output=True, text=True, check=True)
gate_rows = [l.split()[1:] for l in out.stdout.splitlines() if l.startswith("ROW ")]
for parts in gate_rows:
    *label_parts, seconds, n_compiles = parts
    label = " ".join(label_parts)
    print(f"{label:<28s} {float(seconds):6.2f} s   {n_compiles} compiles")
same_line = next(l for l in out.stdout.splitlines() if l.startswith("SAME "))
print("\nsame answer both times:", same_line.split()[1])

first assembly                19.94 s   29 compiles
re-assembled from scratch      7.54 s   2 compiles

same answer both times: True


**PLACEHOLDER -- filled in after execution.**

## 4. The three levels of "warm"

Same solve (`stellarator_helias`, MDF), three regimes:

- **cold** -- fresh process, no persistent cache: JAX traces every jitted block, lowers
  it to HLO, compiles it to native code, and loads the executable. All four steps paid.
- **persistent disk cache** (`--cache`, `run_cold_matrix._enable_compilation_cache`) --
  a fresh process again, so trace and lower still happen, but the *compile* step is a
  cache hit against `jax_compilation_cache_dir` and is skipped.
- **same process, objects retained** -- no fresh process at all. The Python-level jit
  cache already holds the compiled executable keyed on this exact (structurally equal)
  graph, so nothing above the arithmetic runs again.

Each regime is a full solve of the same problem via `session.open_session(...).mdf()` --
subprocesses for the first two, so "fresh process" is real and not simulated.

In [9]:
driver_script = SCRATCH / "warm_level.py"
driver_script.write_text(r'''
import argparse, time
parser = argparse.ArgumentParser()
parser.add_argument("--cache", default=None)
args = parser.parse_args()
import jax
jax.config.update("jax_enable_x64", True)
if args.cache:
    from functional_process.run_cold_matrix import _enable_compilation_cache
    _enable_compilation_cache(args.cache)
from functional_process import session
from functional_process.indat import REFERENCE_INPUT_FILE
t0 = time.perf_counter()
live = session.open_session(REFERENCE_INPUT_FILE)
answer = live.mdf()
t1 = time.perf_counter()
print(f"SECONDS {t1 - t0:.3f}")
''')

def run_subprocess(cache_dir=None):
    args = [PY, str(driver_script)]
    if cache_dir is not None:
        args += ["--cache", str(cache_dir)]
    out = subprocess.run(args, cwd=REPO, capture_output=True, text=True, check=True)
    line = next(l for l in out.stdout.splitlines() if l.startswith("SECONDS"))
    return float(line.split()[1])

cache_dir = SCRATCH / "jax_persistent_cache"
import shutil
shutil.rmtree(cache_dir, ignore_errors=True)
cache_dir.mkdir(parents=True)

cold_seconds = run_subprocess(cache_dir=None)
_populate_seconds = run_subprocess(cache_dir=cache_dir)   # fills the cache, not reported
warm_disk_seconds = run_subprocess(cache_dir=cache_dir)

live2 = session.open_session(REFERENCE_INPUT_FILE)
_ = live2.mdf()                      # pays the compile, in this process
began = time.perf_counter()
_ = live2.mdf()                      # objects retained -- everything above is cached
warm_process_seconds = time.perf_counter() - began

print(f"{'cold':<24s} {cold_seconds:7.2f} s")
print(f"{'persistent disk cache':<24s} {warm_disk_seconds:7.2f} s")
print(f"{'same process, retained':<24s} {warm_process_seconds:7.2f} s")

cold                       19.83 s
persistent disk cache       9.34 s
same process, retained      0.79 s


**PLACEHOLDER -- filled in after execution.**

## 5. The trap

Re-assembling *between* solves throws away level 3, every time -- that is exactly what
section 3 measured (a fresh graph compiles nothing, but still costs the trace-and-lower
work of a full re-assembly, not the near-zero cost of a retained object).
`run_cold_matrix.run_one` re-assembles per call and is the wrong loop to call twice; the
fast path is `functional_process/session.py` -- assemble a `Session` once, then
`.mdf(cold=...)`/`.sand(cold=...)` per point, which only re-seeds and re-primes (10-40 ms,
zero compiles, per that module's own measurements).